# Platform Setup

Create the catalog schemas, reference tables, and baseline Delta tables for the Frankfurt Airspace Complexity Analytics project.

This notebook is create-only. It is designed to bootstrap the Bronze / Silver / Gold structure before historical backfill begins.

## Scope Defaults

This bootstrap notebook locks in the initial project scope:

- focus airport: `EDDF`
- ingestion radius: `150 NM`
- exploratory range: `120-180 NM`
- analysis box:
  - latitude `49.5` to `50.8`
  - longitude `7.8` to `9.6`
- altitude bands:
  - `SFC-FL100`
  - `FL100-FL245`
- complexity windows:
  - `5-minute` Gold metric windows
  - `15-minute` trend windows
- phase-1 complexity components:
  - `traffic_load`
  - `interaction`
  - `flow_structure`

In [ ]:
from pathlib import Path
import yaml

if "spark" not in globals():
    raise RuntimeError("This notebook expects a Spark session, for example inside Databricks.")

default_catalog = "adsb_airport_eta"
catalog = default_catalog

if "dbutils" in globals():
    dbutils.widgets.text("catalog", default_catalog)
    catalog = dbutils.widgets.get("catalog").strip() or default_catalog

repo_root = Path.cwd().resolve().parent
region_config_path = repo_root / "configs" / "region_config.yaml"
pipeline_config_path = repo_root / "configs" / "pipeline_config.yaml"

with region_config_path.open("r", encoding="utf-8") as handle:
    region_config = yaml.safe_load(handle)

with pipeline_config_path.open("r", encoding="utf-8") as handle:
    pipeline_config = yaml.safe_load(handle)

focus_airport = region_config["focus_airport"]
ingestion_radius_nm = int(region_config["ingestion_radius_nm"])
bbox = region_config["regional_bbox"]
complexity_window_minutes = int(pipeline_config["time_windows"]["complexity_window_minutes"])
trend_window_minutes = int(pipeline_config["time_windows"]["trend_window_minutes"])
complexity_components = pipeline_config["complexity_components"]

schemas = [
    "ref",
    "brz_adsb",
    "brz_weather",
    "slv_airspace",
    "gld_airspace",
    "obs",
]

def constraint_exists(table_name: str, constraint_name: str) -> bool:
    property_key = f"delta.constraints.{constraint_name}"
    properties = spark.sql(f"SHOW TBLPROPERTIES {table_name}").collect()
    return any(row.key == property_key for row in properties)


In [ ]:
table_specs = {
    "ref.project_scope": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.ref.project_scope (
            scope_id STRING NOT NULL COMMENT 'Stable logical key for the study scope',
            focus_airport STRING NOT NULL,
            ingestion_radius_nm INT NOT NULL,
            bbox_min_latitude DOUBLE NOT NULL,
            bbox_max_latitude DOUBLE NOT NULL,
            bbox_min_longitude DOUBLE NOT NULL,
            bbox_max_longitude DOUBLE NOT NULL,
            complexity_window_minutes INT NOT NULL,
            trend_window_minutes INT NOT NULL,
            complexity_components STRING NOT NULL,
            created_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        ) USING DELTA
        COMMENT 'Reference scope configuration for the Frankfurt airspace case study.'
        TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
        """,
        "constraints": {
            "ck_ref_project_scope_keys": """
            ALTER TABLE `{catalog}`.ref.project_scope
            ADD CONSTRAINT ck_ref_project_scope_keys
            CHECK (
                scope_id IS NOT NULL AND focus_airport IS NOT NULL AND complexity_components IS NOT NULL AND
                created_at IS NOT NULL AND run_id IS NOT NULL
            )
            """,
            "ck_ref_project_scope_window_bounds": """
            ALTER TABLE `{catalog}`.ref.project_scope
            ADD CONSTRAINT ck_ref_project_scope_window_bounds
            CHECK (
                ingestion_radius_nm > 0 AND complexity_window_minutes > 0 AND trend_window_minutes > 0 AND
                bbox_min_latitude <= bbox_max_latitude AND bbox_min_longitude <= bbox_max_longitude
            )
            """,
        },
    },
    "ref.altitude_bands": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.ref.altitude_bands (
            band_id STRING NOT NULL,
            band_label STRING NOT NULL,
            min_altitude_ft INT NOT NULL,
            max_altitude_ft INT NOT NULL,
            sort_order INT NOT NULL,
            active_flag BOOLEAN NOT NULL,
            created_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        ) USING DELTA
        COMMENT 'Reference altitude bands used by the first Frankfurt complexity model.'
        TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
        """,
        "constraints": {
            "ck_ref_altitude_bands_keys": """
            ALTER TABLE `{catalog}`.ref.altitude_bands
            ADD CONSTRAINT ck_ref_altitude_bands_keys
            CHECK (band_id IS NOT NULL AND band_label IS NOT NULL AND created_at IS NOT NULL AND run_id IS NOT NULL)
            """,
            "ck_ref_altitude_bands_range": """
            ALTER TABLE `{catalog}`.ref.altitude_bands
            ADD CONSTRAINT ck_ref_altitude_bands_range
            CHECK (min_altitude_ft >= 0 AND max_altitude_ft >= min_altitude_ft)
            """,
        },
    },
    "ref.grid_cells": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.ref.grid_cells (
            grid_id STRING NOT NULL,
            min_latitude DOUBLE NOT NULL,
            max_latitude DOUBLE NOT NULL,
            min_longitude DOUBLE NOT NULL,
            max_longitude DOUBLE NOT NULL,
            center_latitude DOUBLE,
            center_longitude DOUBLE,
            grid_resolution_deg DOUBLE,
            active_flag BOOLEAN NOT NULL,
            created_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        ) USING DELTA
        COMMENT 'Reference rectangular grid cells used for Frankfurt hotspot analysis.'
        TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
        """,
        "constraints": {
            "ck_ref_grid_cells_keys": """
            ALTER TABLE `{catalog}`.ref.grid_cells
            ADD CONSTRAINT ck_ref_grid_cells_keys
            CHECK (grid_id IS NOT NULL AND created_at IS NOT NULL AND run_id IS NOT NULL)
            """,
            "ck_ref_grid_cells_bounds": """
            ALTER TABLE `{catalog}`.ref.grid_cells
            ADD CONSTRAINT ck_ref_grid_cells_bounds
            CHECK (
                min_latitude <= max_latitude AND min_longitude <= max_longitude AND
                min_latitude >= -90 AND max_latitude <= 90 AND min_longitude >= -180 AND max_longitude <= 180
            )
            """,
        },
    },
    "brz_adsb.hist_state_vectors": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.brz_adsb.hist_state_vectors (
            time BIGINT NOT NULL COMMENT 'Event time in epoch seconds',
            icao24 STRING NOT NULL,
            callsign STRING,
            lat DOUBLE,
            lon DOUBLE,
            velocity DOUBLE,
            heading DOUBLE,
            vertrate DOUBLE,
            onground BOOLEAN,
            alert BOOLEAN,
            spi BOOLEAN,
            squawk STRING,
            baroaltitude DOUBLE,
            geoaltitude DOUBLE,
            lastposupdate DOUBLE,
            lastcontact DOUBLE,
            hour BIGINT NOT NULL COMMENT 'OpenSky history partition column',
            source_query_start TIMESTAMP,
            source_query_end TIMESTAMP,
            ingested_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        ) USING DELTA
        COMMENT 'Bronze historical state vectors extracted from OpenSky Trino for the Frankfurt study scope.'
        TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
        """,
        "constraints": {
            "ck_brz_hist_state_vectors_keys": """
            ALTER TABLE `{catalog}`.brz_adsb.hist_state_vectors
            ADD CONSTRAINT ck_brz_hist_state_vectors_keys
            CHECK (time IS NOT NULL AND icao24 IS NOT NULL AND hour IS NOT NULL AND ingested_at IS NOT NULL AND run_id IS NOT NULL)
            """,
        },
    },
    "brz_adsb.hist_flights": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.brz_adsb.hist_flights (
            icao24 STRING NOT NULL,
            firstseen BIGINT NOT NULL,
            lastseen BIGINT NOT NULL,
            estdepartureairport STRING,
            estarrivalairport STRING,
            callsign STRING,
            day BIGINT NOT NULL COMMENT 'OpenSky history partition column',
            source_query_start TIMESTAMP,
            source_query_end TIMESTAMP,
            ingested_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        ) USING DELTA
        COMMENT 'Bronze historical flights extracted from OpenSky Trino for Frankfurt context analysis.'
        TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
        """,
        "constraints": {
            "ck_brz_hist_flights_keys": """
            ALTER TABLE `{catalog}`.brz_adsb.hist_flights
            ADD CONSTRAINT ck_brz_hist_flights_keys
            CHECK (
                icao24 IS NOT NULL AND firstseen IS NOT NULL AND lastseen IS NOT NULL AND day IS NOT NULL AND
                ingested_at IS NOT NULL AND run_id IS NOT NULL
            )
            """,
        },
    },
    "brz_adsb.live_states": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.brz_adsb.live_states (
            snapshot_time TIMESTAMP NOT NULL,
            icao24 STRING NOT NULL,
            callsign STRING,
            origin_country STRING,
            latitude DOUBLE,
            longitude DOUBLE,
            baro_altitude_m DOUBLE,
            geo_altitude_m DOUBLE,
            velocity_mps DOUBLE,
            true_track_deg DOUBLE,
            vertical_rate_mps DOUBLE,
            on_ground BOOLEAN,
            scope_airport STRING NOT NULL,
            scope_radius_nm INT NOT NULL,
            ingested_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        ) USING DELTA
        COMMENT 'Optional Bronze live state snapshots used for recent-validation and demo purposes.'
        TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
        """,
        "constraints": {
            "ck_brz_live_states_keys": """
            ALTER TABLE `{catalog}`.brz_adsb.live_states
            ADD CONSTRAINT ck_brz_live_states_keys
            CHECK (
                snapshot_time IS NOT NULL AND icao24 IS NOT NULL AND scope_airport IS NOT NULL AND
                ingested_at IS NOT NULL AND run_id IS NOT NULL
            )
            """,
        },
    },
    "brz_weather.metar_raw": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.brz_weather.metar_raw (
            station_id STRING NOT NULL,
            observation_time TIMESTAMP NOT NULL,
            raw_text STRING NOT NULL,
            wind_speed_kt DOUBLE,
            wind_gust_kt DOUBLE,
            visibility_m DOUBLE,
            ceiling_ft_agl DOUBLE,
            temperature_c DOUBLE,
            source_system STRING NOT NULL,
            ingested_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        ) USING DELTA
        COMMENT 'Reserved Bronze weather table for a later weather-enrichment phase.'
        TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
        """,
        "constraints": {
            "ck_brz_weather_metar_keys": """
            ALTER TABLE `{catalog}`.brz_weather.metar_raw
            ADD CONSTRAINT ck_brz_weather_metar_keys
            CHECK (
                station_id IS NOT NULL AND observation_time IS NOT NULL AND raw_text IS NOT NULL AND
                source_system IS NOT NULL AND ingested_at IS NOT NULL AND run_id IS NOT NULL
            )
            """,
        },
    },
    "slv_airspace.flight_states_clean": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.slv_airspace.flight_states_clean (
            event_time TIMESTAMP NOT NULL,
            time_window_5m TIMESTAMP NOT NULL,
            grid_id STRING NOT NULL,
            icao24 STRING NOT NULL,
            callsign STRING,
            latitude DOUBLE NOT NULL,
            longitude DOUBLE NOT NULL,
            altitude_ft DOUBLE,
            speed_kt DOUBLE,
            heading_deg DOUBLE,
            vertical_rate_fpm DOUBLE,
            altitude_band_id STRING,
            focus_airport STRING NOT NULL,
            source_table STRING NOT NULL,
            ingested_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        ) USING DELTA
        COMMENT 'Silver cleaned state vectors standardized for gridding and complexity feature engineering.'
        TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
        """,
        "constraints": {
            "ck_slv_flight_states_clean_keys": """
            ALTER TABLE `{catalog}`.slv_airspace.flight_states_clean
            ADD CONSTRAINT ck_slv_flight_states_clean_keys
            CHECK (
                event_time IS NOT NULL AND time_window_5m IS NOT NULL AND grid_id IS NOT NULL AND icao24 IS NOT NULL AND
                focus_airport IS NOT NULL AND source_table IS NOT NULL AND ingested_at IS NOT NULL AND run_id IS NOT NULL
            )
            """,
        },
    },
    "gld_airspace.grid_complexity_5m": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.gld_airspace.grid_complexity_5m (
            window_start TIMESTAMP NOT NULL,
            grid_id STRING NOT NULL,
            aircraft_count BIGINT,
            heading_dispersion DOUBLE,
            altitude_dispersion DOUBLE,
            speed_dispersion DOUBLE,
            complexity_score DOUBLE,
            computed_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        ) USING DELTA
        COMMENT 'Gold 5-minute grid-level complexity metrics for the Frankfurt study area.'
        TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
        """,
        "constraints": {
            "ck_gld_grid_complexity_keys": """
            ALTER TABLE `{catalog}`.gld_airspace.grid_complexity_5m
            ADD CONSTRAINT ck_gld_grid_complexity_keys
            CHECK (window_start IS NOT NULL AND grid_id IS NOT NULL AND computed_at IS NOT NULL AND run_id IS NOT NULL)
            """,
        },
    },
    "gld_airspace.complexity_hotspots": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.gld_airspace.complexity_hotspots (
            grid_id STRING NOT NULL,
            avg_complexity_score DOUBLE,
            peak_complexity_score DOUBLE,
            num_high_complexity_windows BIGINT,
            ranking INT,
            computed_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        ) USING DELTA
        COMMENT 'Gold hotspot ranking table for the Frankfurt case study.'
        TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
        """,
        "constraints": {
            "ck_gld_complexity_hotspots_keys": """
            ALTER TABLE `{catalog}`.gld_airspace.complexity_hotspots
            ADD CONSTRAINT ck_gld_complexity_hotspots_keys
            CHECK (grid_id IS NOT NULL AND computed_at IS NOT NULL AND run_id IS NOT NULL)
            """,
        },
    },
    "gld_airspace.complexity_trend_15m": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.gld_airspace.complexity_trend_15m (
            window_start TIMESTAMP NOT NULL,
            avg_complexity_score DOUBLE,
            peak_complexity_score DOUBLE,
            active_grid_count BIGINT,
            active_aircraft_count BIGINT,
            computed_at TIMESTAMP NOT NULL,
            run_id STRING NOT NULL
        ) USING DELTA
        COMMENT 'Gold 15-minute trend table used for dashboard and figure generation.'
        TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
        """,
        "constraints": {
            "ck_gld_complexity_trend_keys": """
            ALTER TABLE `{catalog}`.gld_airspace.complexity_trend_15m
            ADD CONSTRAINT ck_gld_complexity_trend_keys
            CHECK (window_start IS NOT NULL AND computed_at IS NOT NULL AND run_id IS NOT NULL)
            """,
        },
    },
    "obs.ingestion_log": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.obs.ingestion_log (
            run_id STRING NOT NULL,
            source_name STRING NOT NULL,
            target_table STRING NOT NULL,
            source_window_start TIMESTAMP,
            source_window_end TIMESTAMP,
            rows_written BIGINT,
            status STRING NOT NULL,
            measured_at TIMESTAMP NOT NULL,
            metadata_json STRING
        ) USING DELTA
        COMMENT 'Operational log for bounded historical extracts and live snapshot ingestions.'
        TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
        """,
        "constraints": {
            "ck_obs_ingestion_log_keys": """
            ALTER TABLE `{catalog}`.obs.ingestion_log
            ADD CONSTRAINT ck_obs_ingestion_log_keys
            CHECK (
                run_id IS NOT NULL AND source_name IS NOT NULL AND target_table IS NOT NULL AND
                status IS NOT NULL AND measured_at IS NOT NULL
            )
            """,
        },
    },
    "obs.pipeline_run_log": {
        "ddl": """
        CREATE TABLE IF NOT EXISTS `{catalog}`.obs.pipeline_run_log (
            run_id STRING NOT NULL,
            pipeline_name STRING NOT NULL,
            layer STRING NOT NULL,
            target_table STRING NOT NULL,
            status STRING NOT NULL,
            rows_read BIGINT,
            rows_written BIGINT,
            started_at TIMESTAMP NOT NULL,
            completed_at TIMESTAMP,
            error_message STRING,
            metadata_json STRING
        ) USING DELTA
        COMMENT 'Pipeline execution log for the Frankfurt Bronze/Silver/Gold flow.'
        TBLPROPERTIES ('delta.columnMapping.mode' = 'name')
        """,
        "constraints": {
            "ck_obs_pipeline_run_log_keys": """
            ALTER TABLE `{catalog}`.obs.pipeline_run_log
            ADD CONSTRAINT ck_obs_pipeline_run_log_keys
            CHECK (
                run_id IS NOT NULL AND pipeline_name IS NOT NULL AND layer IS NOT NULL AND target_table IS NOT NULL AND
                status IS NOT NULL AND started_at IS NOT NULL
            )
            """,
        },
    },
}


In [ ]:
for schema_name in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.{schema_name}")

tables_ensured = []
constraints_added = []
constraints_existing = []

for table_name, spec in table_specs.items():
    full_name = f"{catalog}.{table_name}"
    spark.sql(spec["ddl"].format(catalog=catalog))
    tables_ensured.append(full_name)

    for constraint_name, constraint_sql in spec["constraints"].items():
        if constraint_exists(full_name, constraint_name):
            constraints_existing.append(f"{full_name}:{constraint_name}")
            continue

        spark.sql(constraint_sql.format(catalog=catalog))
        constraints_added.append(f"{full_name}:{constraint_name}")


In [ ]:
spark.sql(
    f"""
    MERGE INTO `{catalog}`.ref.project_scope AS target
    USING (
        SELECT
            'fra_case_study_v1' AS scope_id,
            '{focus_airport}' AS focus_airport,
            {ingestion_radius_nm} AS ingestion_radius_nm,
            {bbox['min_latitude']} AS bbox_min_latitude,
            {bbox['max_latitude']} AS bbox_max_latitude,
            {bbox['min_longitude']} AS bbox_min_longitude,
            {bbox['max_longitude']} AS bbox_max_longitude,
            {complexity_window_minutes} AS complexity_window_minutes,
            {trend_window_minutes} AS trend_window_minutes,
            '{','.join(complexity_components)}' AS complexity_components
    ) AS source
    ON target.scope_id = source.scope_id
    WHEN MATCHED THEN UPDATE SET
        target.focus_airport = source.focus_airport,
        target.ingestion_radius_nm = source.ingestion_radius_nm,
        target.bbox_min_latitude = source.bbox_min_latitude,
        target.bbox_max_latitude = source.bbox_max_latitude,
        target.bbox_min_longitude = source.bbox_min_longitude,
        target.bbox_max_longitude = source.bbox_max_longitude,
        target.complexity_window_minutes = source.complexity_window_minutes,
        target.trend_window_minutes = source.trend_window_minutes,
        target.complexity_components = source.complexity_components,
        target.created_at = current_timestamp(),
        target.run_id = 'platform_setup'
    WHEN NOT MATCHED THEN INSERT (
        scope_id,
        focus_airport,
        ingestion_radius_nm,
        bbox_min_latitude,
        bbox_max_latitude,
        bbox_min_longitude,
        bbox_max_longitude,
        complexity_window_minutes,
        trend_window_minutes,
        complexity_components,
        created_at,
        run_id
    ) VALUES (
        source.scope_id,
        source.focus_airport,
        source.ingestion_radius_nm,
        source.bbox_min_latitude,
        source.bbox_max_latitude,
        source.bbox_min_longitude,
        source.bbox_max_longitude,
        source.complexity_window_minutes,
        source.trend_window_minutes,
        source.complexity_components,
        current_timestamp(),
        'platform_setup'
    )
    """
)

for sort_order, band in enumerate(region_config["altitude_bands"], start=1):
    spark.sql(
        f"""
        MERGE INTO `{catalog}`.ref.altitude_bands AS target
        USING (
            SELECT
                '{band['band_id']}' AS band_id,
                '{band['band_label']}' AS band_label,
                {int(band['min_altitude_ft'])} AS min_altitude_ft,
                {int(band['max_altitude_ft'])} AS max_altitude_ft,
                {sort_order} AS sort_order,
                TRUE AS active_flag
        ) AS source
        ON target.band_id = source.band_id
        WHEN MATCHED THEN UPDATE SET
            target.band_label = source.band_label,
            target.min_altitude_ft = source.min_altitude_ft,
            target.max_altitude_ft = source.max_altitude_ft,
            target.sort_order = source.sort_order,
            target.active_flag = source.active_flag,
            target.created_at = current_timestamp(),
            target.run_id = 'platform_setup'
        WHEN NOT MATCHED THEN INSERT (
            band_id,
            band_label,
            min_altitude_ft,
            max_altitude_ft,
            sort_order,
            active_flag,
            created_at,
            run_id
        ) VALUES (
            source.band_id,
            source.band_label,
            source.min_altitude_ft,
            source.max_altitude_ft,
            source.sort_order,
            source.active_flag,
            current_timestamp(),
            'platform_setup'
        )
        """
    )


In [ ]:
result = {
    'status': 'success',
    'catalog': catalog,
    'schemas_created': [f"{catalog}.{schema_name}" for schema_name in schemas],
    'tables_ensured': tables_ensured,
    'constraints_added': constraints_added,
    'constraints_existing': constraints_existing,
    'table_count': len(table_specs),
    'scope_defaults': {
        'focus_airport': focus_airport,
        'ingestion_radius_nm': ingestion_radius_nm,
        'bbox': bbox,
        'complexity_window_minutes': complexity_window_minutes,
        'trend_window_minutes': trend_window_minutes,
        'complexity_components': complexity_components,
    },
}

result